In [70]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:95% !important;}
div.cell.code_cell.rendered{width:95%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:22pt;}
.inner_cell{font-size:22pt;}
div.text_cell_render pre code {font-size:22pt; line-height:30px;}
div.output {font-size:20pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:22pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:20pt;padding:5px;}
table.dataframe{font-size:22px;}
</style>
"""))

# 1. tensorflow v2.xx에서 v1 사용하기

In [71]:
import tensorflow.compat.v1 as tf
tf.disable_v2_behavior() # tensorflow v2 비활성화하고 v1만 활성화
import numpy as np
import pandas as pd

## Tensorflow
- 데이터 흐름 그래프(tensor 흐름을 나타내는 설계도)를 사용하는 수치 계산 라이브러리
- 그래프는 node(데이터, 연산)와 edge로 구성
- sess = tf.Session()을 이용하여 실행환경
- sess.run()을 통해서 실행결과를 확인

In [72]:
# 1. tensor(상수node, 변수node, 연산node) 정의
node1 = tf.constant('Hello, Tensorflow')
# 2. 세션 생성(실행하는 환경 생성)
sess = tf.Session()
# 3. 실행
print(sess.run(node1))
print(sess.run(node1).decode())

b'Hello, Tensorflow'
Hello, Tensorflow


In [73]:
# 간단한 연산 tensor 그래프
# 1. 그래프 정의
node1 = tf.constant(10, dtype=tf.float16)
node2 = tf.constant(20, dtype=tf.float16)
node3 = tf.add(node1, node2)
# 2. 세션 생성
sess = tf.Session()
# 3. 세션 실행 및 결과
n1, n2, n3 = sess.run([node1, node2, node3])
print(n1, n2, n3)

10.0 20.0 30.0


In [74]:
# 타입 변경
node1 = tf.constant(np.array([10,20,30]), dtype=tf.int16 )
node2 = tf.cast(node1, dtype=tf.float32)
sess = tf.Session()
print(sess.run( [node1, node2] ))

[array([10, 20, 30], dtype=int16), array([10., 20., 30.], dtype=float32)]


In [75]:
# 평균값 계산 : tf.reduce_mean()
data = np.array([1., 2, 3, 4])
m = tf.reduce_mean(data)
sess = tf.Session()
sess.run(m)

2.5

In [76]:
# tf.random_normal([shape]) : 평균0, 표준편차는 1인 shape 난수 배열 tensor. 기본적으로 float32
w = tf.random.normal([1,3])
sess = tf.Session()
sess.run(w)

array([[ 0.49793735, -0.70907605,  1.2253817 ]], dtype=float32)

In [77]:
# 변수노드
w = tf.Variable( tf.random.normal([1]) )
sess = tf.Session()
sess.run(tf.global_variables_initializer()) # 난수가 발생될 변수 초기화
sess.run(w)

array([1.3460503], dtype=float32)

# 2. tensorflow v1을 이용한 회귀분석 구현
## 2.1 독립(입력)변수 x가 1개, 종속(타겟)변수 y가 1개

In [78]:
# tensor 그래프 정의
# 데이터 셋 확보
x = np.array([1,2,3])
y = np.array([2,3,4])
# weight와 bias
w = tf.Variable( tf.random.normal([1]), name='weight' )
b = tf.Variable( tf.random.normal([1]), name='bias')
# hat, hypothesis : 결과는 numpy배열
H = w * x + b
# cost function (손실함수 : mse) : H-y의 제곱의 평균
cost = tf.reduce_mean(tf.square(H-y))
'''
학습 목적 : cost가 최소가 되는 w와 b를 찾는 것
cost함수가 2차함수이므로 곡선 그래프, 곡선 위 미분값이 0이 되는 방향 학습(경사하강법:GradientDescent)
'''
optimizer = tf.train.GradientDescentOptimizer(learning_rate=0.01)
train = optimizer.minimize(cost)
# 세션 생성
sess = tf.Session()
# w와 b 초기화
sess.run(tf.global_variables_initializer())
# 학습(v2에서의 fit함수)
for step in range(1, 6001):
    _, cost_val, w_val, b_val = sess.run([train, cost, w, b])
    if step%300==1:
        print(f'{step}번째 cost:{cost_val}, w:{w_val}, b:{b_val}')
print(f'{step}번째 cost:{cost_val}, w:{w_val}, b:{b_val}')

1번째 cost:10.32231616973877, w:[-0.3409741], b:[1.0421172]
301번째 cost:0.00953772384673357, w:[0.8868452], b:[1.2572275]
601번째 cost:0.0022505377419292927, w:[0.94503397], b:[1.1249506]
901번째 cost:0.0005310215638019145, w:[0.9733002], b:[1.0606947]
1201번째 cost:0.00012530028470791876, w:[0.98703045], b:[1.0294828]
1501번째 cost:2.9566639568656683e-05, w:[0.9936999], b:[1.0143216]
1801번째 cost:6.97618861522642e-06, w:[0.9969398], b:[1.0069566]
2101번째 cost:1.646350483497372e-06, w:[0.99851334], b:[1.0033795]
2401번째 cost:3.8863745999151433e-07, w:[0.99927765], b:[1.001642]
2701번째 cost:9.186921801074277e-08, w:[0.99964875], b:[1.0007985]
3001번째 cost:2.182878233725205e-08, w:[0.99982876], b:[1.0003891]
3301번째 cost:5.291691085318462e-09, w:[0.9999156], b:[1.0001917]
3601번째 cost:1.31313504514452e-09, w:[0.9999581], b:[1.0000955]
3901번째 cost:3.530213066316179e-10, w:[0.9999785], b:[1.0000495]
4201번째 cost:8.733517381509515e-11, w:[0.9999891], b:[1.0000248]
4501번째 cost:8.733517381509515e-11, w:[0.99998

In [81]:
w_, b_ = sess.run([w[0], b[0]])
w_, b_

(0.9999891, 1.0000248)

In [82]:
def predict(x):
    return x*w_ + b_

In [83]:
predict(5)

5.999970257282257

## 2.2 predict을 위한 placeholder이용
- placeholder : 외부에서 데이터를 입력받을 수 있는 노드

In [84]:
x = tf.placeholder(dtype=np.float32)
H = w_*x + b_
sess = tf.Session()
sess.run([H, x], {x:2.5})

[3.4999976, array(2.5, dtype=float32)]

In [85]:
sess.run(H, {x: np.array([2, 3, 3])})

array([3.0000029, 3.9999921, 3.9999921], dtype=float32)

In [86]:
# tensor 그래프 정의
# 데이터 셋 확보
x_data = np.array([1,2,3])
y_data = np.array([2,3,4])
# placeholder 노드 설정
x = tf.placeholder(dtype=tf.float32)
y = tf.placeholder(dtype=tf.float32)
# weight와 bias
w = tf.Variable( tf.random.normal([1]), name='weight' )
b = tf.Variable( tf.random.normal([1]), name='bias')
# hat, hypothesis : 결과는 numpy배열
H = w * x + b
# cost function (손실함수 : mse) : H-y의 제곱의 평균
cost = tf.reduce_mean(tf.square(H-y))
'''
학습 목적 : cost가 최소가 되는 w와 b를 찾는 것
cost함수가 2차함수이므로 곡선 그래프, 곡선 위 미분값이 0이 되는 방향 학습(경사하강법:GradientDescent)
'''
optimizer = tf.train.GradientDescentOptimizer(learning_rate=0.01)
train = optimizer.minimize(cost)
# 세션 생성
sess = tf.Session()
# w와 b 초기화
sess.run(tf.global_variables_initializer())
# 학습(v2에서의 fit함수)
for step in range(1, 6001):
    _, cost_val, w_val, b_val = sess.run([train, cost, w, b],
                                        feed_dict={x:x_data, y:y_data})
    if step%300==1:
        print(f'{step}번째 cost:{cost_val}, w:{w_val}, b:{b_val}')
print(f'{step}번째 cost:{cost_val}, w:{w_val}, b:{b_val}')

1번째 cost:12.976882934570312, w:[-0.3945461], b:[0.7950175]
301번째 cost:0.003984180744737387, w:[0.9268659], b:[1.1662511]
601번째 cost:0.0009401001152582467, w:[0.9644748], b:[1.0807575]
901번째 cost:0.00022182460816111416, w:[0.9827434], b:[1.0392284]
1201번째 cost:5.234409400145523e-05, w:[0.99161726], b:[1.019056]
1501번째 cost:1.2350108590908349e-05, w:[0.99592817], b:[1.0092562]
1801번째 cost:2.9141283448552713e-06, w:[0.9980221], b:[1.0044962]
2101번째 cost:6.877987175357703e-07, w:[0.9990391], b:[1.0021843]
2401번째 cost:1.6239046374266763e-07, w:[0.99953294], b:[1.0010614]
2701번째 cost:3.852288088523892e-08, w:[0.99977255], b:[1.0005169]
3001번째 cost:9.25397625195501e-09, w:[0.99988854], b:[1.0002534]
3301번째 cost:2.2168364921526518e-09, w:[0.99994546], b:[1.000124]
3601번째 cost:5.897315413783133e-10, w:[0.9999723], b:[1.0000639]
3901번째 cost:1.1361104418350365e-10, w:[0.9999876], b:[1.0000281]
4201번째 cost:8.710306781400945e-11, w:[0.9999891], b:[1.0000248]
4501번째 cost:8.710306781400945e-11, w:[0.

In [88]:
# 예측하기
sess.run(H, feed_dict={x:2.5})

array([3.4999976], dtype=float32)

In [89]:
sess.run(H, feed_dict={x:np.array([2.5, 3.5])})

array([3.4999976, 4.4999866], dtype=float32)

## 2.3 scale이 다른 데이터들의 회귀분석 구현(scale조정X)

In [90]:
# tensor 그래프 정의
# 데이터 셋 확보
x_data = np.array([1,2,5,8,10])
y_data = np.array([5,15,68,80,95])
# placeholder 노드 설정
x = tf.placeholder(dtype=tf.float32)
y = tf.placeholder(dtype=tf.float32)
# weight와 bias
w = tf.Variable( tf.random.normal([1]), name='weight' )
b = tf.Variable( tf.random.normal([1]), name='bias')
# hat, hypothesis : 결과는 numpy배열
H = w * x + b
# cost function (손실함수 : mse) : H-y의 제곱의 평균
cost = tf.reduce_mean(tf.square(H-y))
'''
학습 목적 : cost가 최소가 되는 w와 b를 찾는 것
cost함수가 2차함수이므로 곡선 그래프, 곡선 위 미분값이 0이 되는 방향 학습(경사하강법:GradientDescent)
'''
# optimizer = tf.train.GradientDescentOptimizer(learning_rate=0.01)
# train = optimizer.minimize(cost)
train = tf.train.GradientDescentOptimizer(learning_rate=0.01).minimize(cost)
# 세션 생성
sess = tf.Session()
# w와 b 초기화
sess.run(tf.global_variables_initializer())
# 학습(v2에서의 fit함수)
for step in range(1, 6001):
    _, cost_val, w_val, b_val = sess.run([train, cost, w, b],
                                        feed_dict={x:x_data, y:y_data})
    if step%300==1:
        print(f'{step}번째 cost:{cost_val}')
print(f'{step}번째 cost:{cost_val}')

1번째 cost:4419.7578125
301번째 cost:79.1766128540039
601번째 cost:79.14051055908203
901번째 cost:79.13948822021484
1201번째 cost:79.13947296142578
1501번째 cost:79.13944244384766
1801번째 cost:79.13945007324219
2101번째 cost:79.13947296142578
2401번째 cost:79.13946533203125
2701번째 cost:79.13946533203125
3001번째 cost:79.13946533203125
3301번째 cost:79.13946533203125
3601번째 cost:79.13946533203125
3901번째 cost:79.13946533203125
4201번째 cost:79.13946533203125
4501번째 cost:79.13946533203125
4801번째 cost:79.13946533203125
5101번째 cost:79.13946533203125
5401번째 cost:79.13946533203125
5701번째 cost:79.13946533203125
6000번째 cost:79.13946533203125


## 2.4 scale이 다른 데이터의 회귀분석(scale조정 O)
### scale조정방법 : 모든 데이터를 일정범위내로 조정
- normalization(정규화) : 모든 데이터를 0~1 사이로 조정
                       X - Xmin
    normalization = ──────────────
                     Xmax - Xmin
        * 위의 식보다 라이브러리 추천(sklearn.preprocessing.MinMaxScaler)
        
- standardization(표준화) : 데이터의 평균을 0, 표준편차를 1로 조정
                        X - Xmean
    standardization = ──────────────
                       Xstd(표준편차)
        * 위의 식보다 라이브러리 추천(sklearn.preprocessing.StandardScaler)

In [91]:
# 라이브러리를 쓰지 않고 정규화
x_data = np.array([1,2,5,8,10])
y_data = np.array([5,15,68,80,95])
norm_scaled_x_data = (x_data - x_data.min()) / (x_data.max() - x_data.min())
norm_scaled_y_data = (y_data - y_data.min()) / (y_data.max() - y_data.min())
print(norm_scaled_x_data)
print(norm_scaled_y_data)

[0.         0.11111111 0.44444444 0.77777778 1.        ]
[0.         0.11111111 0.7        0.83333333 1.        ]


In [94]:
# 라이브러리를 사용하여 정규화
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler
x_data = np.array([1,2,5,8,10]).reshape(-1, 1)
y_data = np.array([5,15,68,80,95]).reshape(-1, 1)
scaler_x = MinMaxScaler() # x_data를 변환시킬 객체
scaler_x.fit(x_data)
norm_scaled_x_data = scaler_x.transform(x_data)
scaler_y = MinMaxScaler() # y_data를 변환시킬 객체
# scaler_y.fit(y_data)
# norm_scaled_y_data = scaler_y.transform(y_data)
norm_scaled_y_data = scaler_y.fit_transform(y_data)
np.column_stack([x_data, norm_scaled_x_data, y_data, norm_scaled_y_data])

array([[ 1.        ,  0.        ,  5.        ,  0.        ],
       [ 2.        ,  0.11111111, 15.        ,  0.11111111],
       [ 5.        ,  0.44444444, 68.        ,  0.7       ],
       [ 8.        ,  0.77777778, 80.        ,  0.83333333],
       [10.        ,  1.        , 95.        ,  1.        ]])

In [99]:
# 라이브러리를 쓰지 않고 표준화
x_data = np.array([1,2,5,8,10])
y_data = np.array([5,15,68,80,95])
stan_scaled_x_data = ( x_data - x_data.mean() ) / x_data.std()
stan_scaled_y_data = ( y_data - y_data.mean() ) / y_data.std()
print(np.column_stack([x_data, stan_scaled_x_data, norm_scaled_x_data]))
print()
print(np.column_stack([y_data, stan_scaled_y_data, norm_scaled_y_data]))

[[ 1.         -1.22474487  0.        ]
 [ 2.         -0.93313895  0.11111111]
 [ 5.         -0.05832118  0.44444444]
 [ 8.          0.81649658  0.77777778]
 [10.          1.39970842  1.        ]]

[[ 5.         -1.32373476  0.        ]
 [15.         -1.04563922  0.11111111]
 [68.          0.42826713  0.7       ]
 [80.          0.76198177  0.83333333]
 [95.          1.17912508  1.        ]]


In [101]:
# 라이브러리를 사용하여 표준화
x_data = np.array([1,2,5,8,10]).reshape(-1,1)
y_data = np.array([5,15,68,80,95]).reshape(-1,1)
scaler_x = StandardScaler()
stan_scaled_x_data = scaler_x.fit_transform(x_data)
scaler_y = StandardScaler()
stan_scaled_y_data = scaler_y.fit_transform(y_data)
np.column_stack([stan_scaled_x_data, stan_scaled_y_data])

array([[-1.22474487, -1.32373476],
       [-0.93313895, -1.04563922],
       [-0.05832118,  0.42826713],
       [ 0.81649658,  0.76198177],
       [ 1.39970842,  1.17912508]])

In [102]:
# 스케일 조정된 데이터를 다시 복구 : inverse_transform() 이용
scaler_x.inverse_transform(stan_scaled_x_data)

array([[ 1.],
       [ 2.],
       [ 5.],
       [ 8.],
       [10.]])

In [103]:
scaler_y.inverse_transform(stan_scaled_y_data)

array([[ 5.],
       [15.],
       [68.],
       [80.],
       [95.]])

In [104]:
# tensor 그래프 정의
# 데이터 셋 확보
x_data = np.array([1,2,5,8,10])
y_data = np.array([5,15,68,80,95])
# placeholder 노드 설정
x = tf.placeholder(dtype=tf.float32)
y = tf.placeholder(dtype=tf.float32)
# weight와 bias
w = tf.Variable( tf.random.normal([1]), name='weight' )
b = tf.Variable( tf.random.normal([1]), name='bias')
# hat, hypothesis : 결과는 numpy배열
H = w * x + b
# cost function (손실함수 : mse) : H-y의 제곱의 평균
cost = tf.reduce_mean(tf.square(H-y))
'''
학습 목적 : cost가 최소가 되는 w와 b를 찾는 것
cost함수가 2차함수이므로 곡선 그래프, 곡선 위 미분값이 0이 되는 방향 학습(경사하강법:GradientDescent)
'''
# optimizer = tf.train.GradientDescentOptimizer(learning_rate=0.01)
# train = optimizer.minimize(cost)
train = tf.train.GradientDescentOptimizer(learning_rate=0.01).minimize(cost)
# 세션 생성
sess = tf.Session()
# w와 b 초기화
sess.run(tf.global_variables_initializer())
# 학습(v2에서의 fit함수)
for step in range(1, 2001):
    _, cost_val = sess.run([train, cost],
                        feed_dict={x:norm_scaled_x_data,
                                   y:norm_scaled_y_data})
    if step%300==1:
        print(f'{step}번째 cost:{cost_val}')
print(f'{step}번째 cost:{cost_val}')

1번째 cost:0.32015612721443176
301번째 cost:0.08128377050161362
601번째 cost:0.027417009696364403
901번째 cost:0.014124820940196514
1201번째 cost:0.010844824835658073
1501번째 cost:0.010035449638962746
1801번째 cost:0.00983573030680418
2000번째 cost:0.009796163998544216
